# Week 2 Exploratory Data Analysis and Visualization
## Titanic Survival Prediction Dataset

This notebook is built specifically for the cleaned Titanic dataset file named titanic_cleaned.csv.

Before running, place titanic_cleaned.csv in the same folder as this notebook. Run all cells from top to bottom. Charts are saved in the outputs folder for the Word report.

## 1 Import Libraries

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Libraries imported successfully.")

## 2 Load Dataset

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("titanic_cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "titanic_cleaned.csv was not found. Put it in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape (rows, columns):", df.shape)
print("Columns:", list(df.columns))
display(df.head())

## 3 Initial Dataset Exploration

In [ ]:
print("DATA TYPES AND NON NULL COUNTS")
df.info()

print("\nMISSING VALUES")
missing_before = df.isnull().sum().sort_values(ascending=False)
display(missing_before[missing_before > 0])

print("\nDUPLICATE ROWS:", df.duplicated().sum())

print("\nNUMERICAL SUMMARY")
display(df.describe().T)

print("\nCATEGORICAL SUMMARY")
display(df.describe(include="object").T)

## 4 Data Preparation for EDA

The dataset is already cleaned. This step creates AgeGroup from Age for deeper comparison. The existing features Deck, Title, FamilySize, IsAlone, and FareLog are used directly.

In [ ]:
# Standardise categorical fields
df["Sex"] = df["Sex"].astype("string").str.strip().str.title()
df["Embarked"] = df["Embarked"].astype("string").str.strip().str.title()

# Create age groups for EDA
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 19, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"],
    include_lowest=True
)

print("Data preparation completed successfully.")
print("Total missing values:", df.isnull().sum().sum())
display(df.head())

## 5 Visualisation 1 Passenger Count by Class

In [ ]:
class_count = df["Pclass"].value_counts().sort_index()

plt.figure(figsize=(8, 4.5))
ax = sns.barplot(x=class_count.index, y=class_count.values, color="#2E86C1")
plt.title("Passenger Count by Travel Class", fontweight="bold")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_passenger_count_by_class.png", dpi=300, bbox_inches="tight")
plt.show()

display(class_count)

## 6 Visualisation 2 Overall Survival Count

In [ ]:
survival_counts = df["Survived"].value_counts().sort_index()

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="Survived", hue="Survived", palette=["#E74C3C", "#2ECC71"], legend=False)
plt.title("Number of Passengers Who Survived", fontweight="bold")
plt.xlabel("Survival Status")
plt.ylabel("Number of Passengers")
plt.xticks([0, 1], ["Did Not Survive", "Survived"])
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_survival_count.png", dpi=300, bbox_inches="tight")
plt.show()

print("Passengers who did not survive:", survival_counts[0])
print("Passengers who survived:", survival_counts[1])
print("Overall survival rate:", round(df["Survived"].mean() * 100, 2), "%")

## 7 Visualisation 3 Survival by Gender

In [ ]:
gender_summary = df.groupby("Sex")["Survived"].agg(Passengers="count", Survivors="sum", SurvivalRate="mean")
gender_summary["SurvivalRatePercent"] = (gender_summary["SurvivalRate"] * 100).round(2)

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="Sex", hue="Survived", palette=["#E74C3C", "#2ECC71"])
plt.title("Survival Status by Gender", fontweight="bold")
plt.xlabel("Gender")
plt.ylabel("Number of Passengers")
plt.legend(title="Survived", labels=["No", "Yes"])
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_survival_by_gender.png", dpi=300, bbox_inches="tight")
plt.show()

display(gender_summary)

## 8 Visualisation 4 Survival by Passenger Class

In [ ]:
class_summary = df.groupby("Pclass")["Survived"].agg(Passengers="count", Survivors="sum", SurvivalRate="mean")
class_summary["SurvivalRatePercent"] = (class_summary["SurvivalRate"] * 100).round(2)

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x="Pclass", hue="Survived", palette=["#E74C3C", "#2ECC71"])
plt.title("Survival Status by Passenger Class", fontweight="bold")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
plt.legend(title="Survived", labels=["No", "Yes"])
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_survival_by_class.png", dpi=300, bbox_inches="tight")
plt.show()

display(class_summary)

## 9 Visualisation 5 Age Distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="Age", hue="Survived", bins=30, kde=True, palette=["#E74C3C", "#2ECC71"], alpha=0.55)
plt.title("Age Distribution by Survival Status", fontweight="bold")
plt.xlabel("Age in Years")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_age_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

display(df.groupby("Survived")["Age"].agg(["count", "mean", "median"]).round(2))

## 10 Visualisation 6 Fare Distribution and Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.histplot(df["Fare"], bins=40, kde=True, color="#3498DB", ax=axes[0])
axes[0].set_title("Fare Distribution", fontweight="bold")
axes[0].set_xlabel("Fare Paid")
axes[0].set_ylabel("Number of Passengers")

sns.boxplot(x=df["Fare"], color="#F5B041", ax=axes[1])
axes[1].set_title("Fare Outliers", fontweight="bold")
axes[1].set_xlabel("Fare Paid")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_fare_distribution_outliers.png", dpi=300, bbox_inches="tight")
plt.show()

print("Average fare:", round(df["Fare"].mean(), 2))
print("Median fare:", round(df["Fare"].median(), 2))
print("Maximum fare:", round(df["Fare"].max(), 2))

## 11 Visualisation 7 Survival Rate by Age Group

In [ ]:
age_group_summary = df.groupby("AgeGroup", observed=False)["Survived"].agg(Passengers="count", SurvivalRate="mean").reset_index()
age_group_summary["SurvivalRatePercent"] = (age_group_summary["SurvivalRate"] * 100).round(2)

plt.figure(figsize=(9, 5))
ax = sns.barplot(data=age_group_summary, x="AgeGroup", y="SurvivalRatePercent", color="#9B59B6")
plt.title("Survival Rate by Age Group", fontweight="bold")
plt.xlabel("Age Group")
plt.ylabel("Survival Rate Percentage")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "07_survival_by_agegroup.png", dpi=300, bbox_inches="tight")
plt.show()

display(age_group_summary)

## 12 Visualisation 8 Survival Rate by Embarkation Port

In [ ]:
embarked_summary = df.groupby("Embarked")["Survived"].agg(Passengers="count", SurvivalRate="mean").reset_index()
embarked_summary["SurvivalRatePercent"] = (embarked_summary["SurvivalRate"] * 100).round(2)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=embarked_summary, x="Embarked", y="SurvivalRatePercent", hue="Embarked", palette="viridis", legend=False)
plt.title("Survival Rate by Embarkation Port", fontweight="bold")
plt.xlabel("Port (C = Cherbourg, Q = Queenstown, S = Southampton)")
plt.ylabel("Survival Rate Percentage")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "08_survival_by_embarked.png", dpi=300, bbox_inches="tight")
plt.show()

display(embarked_summary)

## 13 Visualisation 9 Correlation Heatmap

In [ ]:
numeric_columns = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]

plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_columns].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Correlation Heatmap of Numerical Features", fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "09_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 14 Visualisation 10 Family Size and Survival

In [ ]:
family_summary = df.groupby("FamilySize")["Survived"].agg(Passengers="count", SurvivalRate="mean").reset_index()
family_summary["SurvivalRatePercent"] = (family_summary["SurvivalRate"] * 100).round(2)

plt.figure(figsize=(10, 5))
sns.lineplot(data=family_summary, x="FamilySize", y="SurvivalRatePercent", marker="o", color="#16A085", linewidth=2.5)
plt.title("Survival Rate by Family Size", fontweight="bold")
plt.xlabel("Family Size")
plt.ylabel("Survival Rate Percentage")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "10_survival_by_familysize.png", dpi=300, bbox_inches="tight")
plt.show()

display(family_summary)

## 15 Automatically Generated Findings

In [ ]:
best_gender = gender_summary["SurvivalRatePercent"].idxmax()
best_gender_rate = gender_summary.loc[best_gender, "SurvivalRatePercent"]
best_class = class_summary["SurvivalRatePercent"].idxmax()
best_class_rate = class_summary.loc[best_class, "SurvivalRatePercent"]
best_port = embarked_summary.loc[embarked_summary["SurvivalRatePercent"].idxmax(), "Embarked"]
best_port_rate = embarked_summary["SurvivalRatePercent"].max()
largest_age_group = age_group_summary.loc[age_group_summary["Passengers"].idxmax(), "AgeGroup"]

print("WEEK 2 EDA FINDINGS")
print("- Overall survival rate was", round(df["Survived"].mean() * 100, 2), "%.")
print(f"- {best_gender} passengers had the highest gender-based survival rate: {best_gender_rate}%.")
print(f"- Passenger class {best_class} had the highest survival rate: {best_class_rate}%.")
print(f"- Embarkation port {best_port} had the highest survival rate: {best_port_rate:.2f}%.")
print(f"- The largest passenger age group was {largest_age_group}.")
print(f"- The median ticket fare was {df['Fare'].median():.2f}, while the maximum was {df['Fare'].max():.2f}; this confirms a right-skewed fare distribution.")
print("- The outputs folder now contains all chart images for the Word report.")

## 16 Save Final EDA Dataset

In [ ]:
df.to_csv("titanic_week2_eda_prepared.csv", index=False)
print("Saved: titanic_week2_eda_prepared.csv")
print("Saved charts folder:", OUTPUT_DIR.resolve())